# Close All vBill Subscriber Accounts

This notebook:
1. Generates an OAuth2 access token (password grant)
2. Loads the list of deactivated vBill accounts from the `bi_datastore.billing_account` MySQL table
3. Iterates through each account and closes it via `DELETE /rest/SubscriberService/v1/subscribers/{accountNumber}`

**Safety:** `DRY_RUN = True` by default. The notebook will list what *would* be closed without actually closing anything. Flip it to `False` once you've reviewed the list.

## 1. Configuration

Set your host, OAuth credentials, and the dry-run flag here.

In [6]:
import os
import time
import requests
from requests.exceptions import RequestException
from sqlalchemy import create_engine
from datetime import datetime, timedelta
import pandas as pd
import logging

# --- API host ---
HOST = os.getenv("API_HOST", "https://sandbox-sg.onebillsoftware.com")

# --- OAuth2 password-grant credentials ---
TOKEN_URL = f"{HOST}/oauth/token"   # <- adjust path if your token endpoint differs
OAUTH_USERNAME = os.getenv("API_USERNAME")
OAUTH_PASSWORD = os.getenv("API_PASSWORD")
OAUTH_CLIENT_ID = os.getenv("CLIENT_ID")
OAUTH_CLIENT_SECRET = os.getenv("CLIENT_SECRET")  # leave blank if not required           

# --- Behaviour ---
DRY_RUN = True              # Set to False to actually close accounts
REQUEST_TIMEOUT = 30          # seconds
MAX_WORKERS = 10              # parallel close requests. Bump to 20-30 if API tolerates it.

# Optional: only close accounts matching this filter. Leave as None to close ALL.
# Example: lambda s: s["accountType"] == 1001
ACCOUNT_FILTER = None

SUBSCRIBERS_URL = f"{HOST}/rest/SubscriberService/v1/subscribers"

BI_DATASTORE_URL = (
    f"mysql+mysqlconnector://{os.environ['DB_USERNAME']}:{os.environ['DB_PASSWORD']}"
    f"@{os.environ['DB_HOST']}/bi_datastore"
)

PROXY_ACCOUNT_NUMBER = os.getenv("DELETION_PROXY_ACCOUNT_NUMBER")

# --- Logging ---
log_filename = f'migration_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler(log_filename), logging.StreamHandler()],
)
logger = logging.getLogger(__name__)

print(f"Host:    {HOST}")
print(f"Dry run: {DRY_RUN}")

Host:    https://sandbox-sg.onebillsoftware.com
Dry run: True


## 2. Generate access token

Calls the token endpoint with `grant_type=password`. The token is cached so the rest of the notebook reuses it.

In [7]:
def fetch_access_token():
    """Get an OAuth2 access token using the password grant."""
    data = {
        "grant_type": "password",
        "username": OAUTH_USERNAME,
        "password": OAUTH_PASSWORD,
        "client_id": OAUTH_CLIENT_ID,
    }
    if OAUTH_CLIENT_SECRET:
        data["client_secret"] = OAUTH_CLIENT_SECRET

    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Accept": "application/json"
    }

    resp = requests.post(TOKEN_URL, data=data, headers=headers, timeout=REQUEST_TIMEOUT)
    if resp.status_code != 200:
        raise RuntimeError(
            f"Token request failed: HTTP {resp.status_code} - {resp.text[:300]}"
        )
    payload = resp.json()
    token = payload.get("access_token")
    if not token:
        raise RuntimeError(f"No access_token in response: {payload}")

    expires_in = payload.get("expires_in")
    token_type = payload.get("token_type", "Bearer")
    if expires_in:
        print(f"Got {token_type} token. Expires in: {expires_in}s")
    else:
        print(f"Got {token_type} token.")
    return token, token_type

ACCESS_TOKEN, TOKEN_TYPE = fetch_access_token()

HEADERS = {
    "Authorization": f"{TOKEN_TYPE} {ACCESS_TOKEN}",
    "Accept": "application/json",
    "Content-Type": "application/json",
    "proxy_accountNumber": os.environ["DELETION_PROXY_ACCOUNT_NUMBER"]
}

Got bearer token. Expires in: 1521s


## 3. Load accounts from MySQL

Pull every deactivated vBill account from `bi_datastore.billing_account`. The notebook will close each `AccountCode` returned here.

In [8]:
ACCOUNT_QUERY = """
SELECT 
	*
FROM 
	bi_datastore.billing_account
WHERE 
	`_DataSource` = 'vBill'
AND 
	`Status` = 'DEACTIVATED'
""".strip()

engine = create_engine(BI_DATASTORE_URL)
df_accounts = pd.read_sql(ACCOUNT_QUERY, con=engine)

# AccountCode as string for consistent dict lookups against Dataverse-side strings
df_accounts["AccountCode"] = df_accounts["AccountCode"].astype(str)

df_accounts["AccountCode_Batch"] = df_accounts["AccountCode"] + "_" + PROXY_ACCOUNT_NUMBER

logger.info(f"Loaded {len(df_accounts):,} accounts from MySQL")
df_accounts.head()

2026-05-19 14:24:05,261 [INFO] Loaded 11,065 accounts from MySQL


,_rowid,_rowmodified,_sourceid,_DataSource,AccountCode,AccountName,AlternateAccountCode,BillToAccountCode,BillToAccountName,CreatedDate,...,_temporary_crmonly_addr2,_temporary_crmonly_suburb,_temporary_crmonly_city,_temporary_crmonly_postcode,_temporary_crmonly_countryiso,_temporary_crmonly_phone_home,_temporary_crmonly_phone_work,_temporary_crmonly_mobile,_temporary_crmonly_dob,AccountCode_Batch
0,181219,2025-05-24 04:30:21,6,vBill,1000000008,Inomial,None,None,None,2005-12-08,...,None,None,None,None,None,None,None,None,None,1000000008_31802
1,104399,2025-05-24 04:32:58,18,vBill,1000000024,"Sales, Cash",None,None,None,2005-12-22,...,None,None,None,None,None,None,None,None,None,1000000024_31802
2,251068597,2022-06-21 18:59:39,219316,vBill,1099978402,Jay Gandhi (Operator),None,None,None,2021-10-27,...,None,Wallaceville,Upper Hutt,5018,NZ,+642040070004,None,+642040070004,1996-06-05,1099978402_31802
3,132819,2021-04-04 14:19:48,10319,vBill,20115569,The Total Saver Limited (In Liquidation) (ICMS),None,None,None,2018-06-14,...,None,Drury,,2577,None,None,+6499730974,None,None,20115569_31802
4,156379,2021-04-04 14:20:00,10320,vBill,20241342,Pan Pacific Travel Corporation Limited,None,None,None,2018-06-14,...,None,Remuera,,,None,None,+6495209193,,None,20241342_31802


## 4. Preview the accounts to be closed

In [10]:
# Apply optional filter (set ACCOUNT_FILTER above to narrow the list)
if ACCOUNT_FILTER is not None:
    df_to_close = df_accounts[df_accounts.apply(ACCOUNT_FILTER, axis=1)].copy()
else:
    df_to_close = df_accounts.copy()

# Drop rows without a usable AccountCode
df_to_close = df_to_close[df_to_close["AccountCode"].notna() & (df_to_close["AccountCode"].astype(str).str.strip() != "")]

logger.info(f"{len(df_to_close):,} accounts queued for closure")
df_to_close[["AccountCode_Batch", "AccountName"]].head(20)

2026-05-19 14:24:12,686 [INFO] 11,065 accounts queued for closure


,AccountCode_Batch,AccountName
0,1000000008_31802,Inomial
1,1000000024_31802,"Sales, Cash"
2,1099978402_31802,Jay Gandhi (Operator)
3,20115569_31802,The Total Saver Limited (In Liquidation) (ICMS)
4,20241342_31802,Pan Pacific Travel Corporation Limited
5,20668769_31802,Yukfoo Limited (X)
6,21186140_31802,Humes Pipeline Systems - Oasis
7,21195013_31802,Evolution IT Limited
8,21223405_31802,Chiroworks Limited
9,21250758_31802,Net Service Limited (X)


## 5. Close the accounts


In [11]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

session = requests.Session()
session.headers.update(HEADERS)

def close_account(account_number: str):
    url = f"{HOST}/rest/SubscriberService/v1/subscribers/{account_number}"
    try:
        resp = session.delete(url, timeout=REQUEST_TIMEOUT)
        if 200 <= resp.status_code < 300:
            return True, f"HTTP {resp.status_code}"
        return False, f"HTTP {resp.status_code}: {resp.text[:200]}"
    except RequestException as e:
        return False, f"Exception: {e}"

results = {"closed": [], "failed": [], "skipped": []}
overall_start = time.time()

# Build the list of (accountCode_Batch, accountName) tuples from the DataFrame
accounts_to_close = [
    (str(row["AccountCode_Batch"]).strip(), str(row.get("AccountName", "") or ""))
    for _, row in df_to_close.iterrows()
    if str(row["AccountCode_Batch"]).strip()
]

# Anything in df_accounts but not in df_to_close (filtered out) -> skipped
skipped_codes = set(df_accounts["AccountCode_Batch"].astype(str)) - {a for a, _ in accounts_to_close}
for code in skipped_codes:
    results["skipped"].append(code)

total = len(accounts_to_close)

if total == 0:
    logger.info("No accounts to close. Done.")
else:
    logger.info(f"Closing {total} accounts with {MAX_WORKERS} workers (DRY_RUN={DRY_RUN})")

    if DRY_RUN:
        for acct, name in accounts_to_close:
            print(f"  DRY-RUN would close {acct} ({name})")
        print(f"\nDry run complete: {total} accounts would have been closed.")
    else:
        progress = {"done": 0}
        progress_lock = Lock()

        def worker(acct, name):
            ok, detail = close_account(acct)
            with progress_lock:
                progress["done"] += 1
                status = "CLOSED" if ok else "FAILED"
                msg = f"[{progress['done']}/{total}] {status}  {acct} ({name}) - {detail}"
                if ok:
                    logger.info(msg)
                else:
                    logger.warning(msg)
            return acct, ok, detail

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = [executor.submit(worker, acct, name) for acct, name in accounts_to_close]
            for fut in as_completed(futures):
                acct, ok, detail = fut.result()
                if ok:
                    results["closed"].append(acct)
                else:
                    results["failed"].append({"accountNumber": acct, "error": detail})

elapsed = time.time() - overall_start
logger.info(
    f"=== Overall: {elapsed:.1f}s, closed {len(results['closed'])}, "
    f"failed {len(results['failed'])}, skipped {len(results['skipped'])} ==="
)

2026-05-19 14:24:36,326 [INFO] Closing 11065 accounts with 10 workers (DRY_RUN=True)
2026-05-19 14:24:36,407 [INFO] === Overall: 0.5s, closed 0, failed 0, skipped 0 ===


  DRY-RUN would close 1000000008_31802 (Inomial)
  DRY-RUN would close 1000000024_31802 (Sales, Cash)
  DRY-RUN would close 1099978402_31802 (Jay Gandhi (Operator))
  DRY-RUN would close 20115569_31802 (The Total Saver Limited (In Liquidation) (ICMS))
  DRY-RUN would close 20241342_31802 (Pan Pacific Travel Corporation Limited)
  DRY-RUN would close 20668769_31802 (Yukfoo Limited (X))
  DRY-RUN would close 21186140_31802 (Humes Pipeline Systems - Oasis)
  DRY-RUN would close 21195013_31802 (Evolution IT Limited)
  DRY-RUN would close 21223405_31802 (Chiroworks Limited)
  DRY-RUN would close 21250758_31802 (Net Service Limited (X))
  DRY-RUN would close 21983007_31802 (Navigatus Consulting (X))
  DRY-RUN would close 22153292_31802 (Method Studios Limited (X))
  DRY-RUN would close 23373357_31802 (M & C Consulting Limited)
  DRY-RUN would close 23440281_31802 (z-BDO New Zealand)
  DRY-RUN would close 24300701_31802 (Omni Tech (ICMS))
  DRY-RUN would close 25323445_31802 (Interlace Techno

## 6. Summary of failures (if any)

In [12]:
if not DRY_RUN and results["failed"]:
    print("The following accounts failed to close:\n")
    for f in results["failed"]:
        print(f"  {f['accountNumber']}: {f['error']}")
else:
    print("No failures to report.")

No failures to report.


## 7. (Optional) Verify by re-fetching

In [13]:
# Re-check by counting closed accounts vs the failed list
if not DRY_RUN:
    logger.info(f"Closed: {len(results['closed']):,}")
    logger.info(f"Failed: {len(results['failed']):,}")
    logger.info(f"Skipped: {len(results['skipped']):,}")
    logger.info(f"Total processed: {len(results['closed']) + len(results['failed']):,} / {len(df_accounts):,}")
else:
    print("Dry run was on; skipping verification.")

Dry run was on; skipping verification.
